In [102]:
#load env variables 
from dotenv import load_dotenv
load_dotenv()

True

In [103]:
# create the loader 
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("paper.pdf")
type(loader)

# actually read the pdf
docs=loader.load()  
type(docs),len(docs),type(docs[0]),docs[0].metadata

(list,
 15,
 langchain_core.documents.base.Document,
 {'producer': 'pdfTeX-1.40.25',
  'creator': 'LaTeX with hyperref',
  'creationdate': '2024-04-10T21:11:43+00:00',
  'source': 'paper.pdf',
  'file_path': 'paper.pdf',
  'total_pages': 15,
  'format': 'PDF 1.5',
  'title': '',
  'author': '',
  'subject': '',
  'keywords': '',
  'moddate': '2024-04-10T21:11:43+00:00',
  'trapped': '',
  'modDate': 'D:20240410211143Z',
  'creationDate': 'D:20240410211143Z',
  'page': 0})

In [104]:
# create splitter 
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
type(splitter)

#actually split
chunks=splitter.split_documents(docs) 
print(type(chunks)),print(len(chunks)),print(chunks[0].metadata)

<class 'list'>
94
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'source': 'paper.pdf', 'file_path': 'paper.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'trapped': '', 'modDate': 'D:20240410211143Z', 'creationDate': 'D:20240410211143Z', 'page': 0}


(None, None, None)

In [105]:
# create embedding model
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# test the embedding model 
vector=embeddings.embed_query("what is transformer")
print(type(vector))

print(vector)
print(len(vector))

#will print dimension of vector or no of words in a sentence
# dimension or no of words depends on embedding model used 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4015.22it/s]


<class 'list'>
[-0.1854417473077774, 0.04515491798520088, -0.026022404432296753, -0.0036607314832508564, 0.022438116371631622, -0.016202835366129875, -0.04091993346810341, 0.07431329786777496, 0.052100397646427155, -0.015700768679380417, -0.015121008269488811, 0.04672654718160629, 0.030482813715934753, 0.044781673699617386, -0.020520301535725594, 0.01667211577296257, -0.05998294800519943, 0.020185714587569237, -0.07053320109844208, -0.09665405005216599, -0.03314298018813133, 0.07608352601528168, -0.06771989166736603, -0.02875690907239914, 0.04429774358868599, -0.013640015386044979, 0.031212197616696358, -0.0586247444152832, -0.05167585238814354, -0.021882962435483932, -0.02064366452395916, -0.041383977979421616, -0.09227176755666733, 0.07309770584106445, -0.10471798479557037, 0.055586062371730804, -0.010495931841433048, -0.01680683344602585, 0.02322673238813877, 0.013393099419772625, 0.0379340685904026, -0.08773507177829742, 0.020271698012948036, -0.057363998144865036, -0.0447370111942

In [106]:
# create chromadb
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print(type(vectorstore))

<class 'langchain_chroma.vectorstores.Chroma'>


In [107]:
# create the retriever - another object that knows how to search.
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4} # gives top 4 relevant chunks
)

#search and find nearest vectors
retrieved_docs = retriever.invoke("what is transformer?")
len(retrieved_docs)

4

In [108]:
# inspect each chunk
for i,doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1}") # or ("chunk ",i+1)
    print(doc.metadata)
    print(doc.page_content[:400],"\n")

Chunk 1
{'subject': '', 'creationDate': 'D:20240410211143Z', 'file_path': 'paper.pdf', 'keywords': '', 'page': 2, 'title': '', 'source': 'paper.pdf', 'moddate': '2024-04-10T21:11:43+00:00', 'format': 'PDF 1.5', 'trapped': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'total_pages': 15, 'author': '', 'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'modDate': 'D:20240410211143Z'}
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1
Encoder and Decoder Stacks
Encoder:
The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The f 

Chunk 2
{'creationDate': 'D:20240410211143Z', 'keywords': '', 'creator': 'LaTeX with hyperref', 'file_path': 'paper.pdf', 'title': '', 'subject': '', 'author': '', 'source': 'paper.pdf', 'modDate':

In [109]:
# create llm
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant",
    temperature=0,
)

In [110]:
# create prompt template
from langchain_core.prompts import PromptTemplate

prompt= PromptTemplate(
    template="""
    You are an ai research assistant.
    answer only from provided context
    if answer is not present, reply:
    "i don't know."

    Context:
    {context}

    Question:
    {question}

    Answer:
""",
    input_variables=["context","question"]
)

In [111]:
# create str output parser 
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [112]:
# convert documents to text
# retriever returns a list of Document objects, but the prompt needs plain text.
def format_docs(docs):
    return "\n\n".join( # join with 2 line break in between 
        doc.page_content for doc in docs
    )

In [ ]:
# build a rag chain
from langchain_core.runnables import RunnablePassthrough

chain=(

    {
        "context": retriever | format_docs,
        "question":RunnablePassthrough()
    } | prompt | llm | parser
    
)

In [119]:
response = chain.invoke(
    "What is transformer ?"
)

print(response)

The Transformer is a model architecture, as shown in Figure 1, that follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.
